# Image Classification with SVM and Feature Extraction

This notebook builds a classical image classification pipeline using handcrafted features and a Support Vector Machine (SVM).

It extracts:

- **HOG** features for shape and edge structure
- **Color histograms** for color distribution
- **Local Binary Pattern (LBP)** histograms for texture

Expected dataset layout:

```text
data/
  train/
    class_a/
      image_001.jpg
      image_002.jpg
    class_b/
      image_001.jpg
      image_002.jpg
```

You can change `DATA_DIR` below if your images are somewhere else.

## 1. Setup

Install the required packages if needed:

```bash
pip install numpy matplotlib scikit-image scikit-learn joblib
```

In [ ]:
from pathlib import Path
import random

import joblib
import matplotlib.pyplot as plt
import numpy as np

from skimage.color import rgb2gray
from skimage.feature import hog, local_binary_pattern
from skimage.io import imread
from skimage.transform import resize

from sklearn.metrics import ConfusionMatrixDisplay, classification_report, accuracy_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("data/train")
IMAGE_SIZE = (128, 128)
MODEL_PATH = Path("svm_image_classifier.joblib")

## 2. Load Images

The loader expects one subfolder per class. Folder names become labels.

In [ ]:
def list_image_files(data_dir):
    image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    files = []

    for class_dir in sorted(p for p in data_dir.iterdir() if p.is_dir()):
        for image_path in sorted(class_dir.rglob("*")):
            if image_path.suffix.lower() in image_extensions:
                files.append((image_path, class_dir.name))

    return files


def read_rgb_image(image_path, image_size=IMAGE_SIZE):
    image = imread(image_path)

    if image.ndim == 2:
        image = np.stack([image, image, image], axis=-1)
    elif image.shape[-1] == 4:
        image = image[..., :3]

    image = resize(
        image,
        (*image_size, 3),
        anti_aliasing=True,
        preserve_range=True,
    )

    return image.astype(np.float32) / 255.0


image_files = list_image_files(DATA_DIR)

if not image_files:
    raise FileNotFoundError(
        f"No images found in {DATA_DIR.resolve()}. Create class folders like data/train/cat/*.jpg."
    )

print(f"Found {len(image_files)} images across {len(set(label for _, label in image_files))} classes.")
print("Classes:", sorted(set(label for _, label in image_files)))

## 3. Preview Samples

In [ ]:
def show_samples(image_files, n=8):
    sample = random.sample(image_files, min(n, len(image_files)))
    cols = min(4, len(sample))
    rows = int(np.ceil(len(sample) / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    axes = np.atleast_1d(axes).ravel()

    for ax, (image_path, label) in zip(axes, sample):
        ax.imshow(read_rgb_image(image_path))
        ax.set_title(label)
        ax.axis("off")

    for ax in axes[len(sample):]:
        ax.axis("off")

    plt.tight_layout()


show_samples(image_files)

## 4. Feature Extraction

The final feature vector concatenates HOG, color histogram, and LBP texture features.

In [ ]:
def extract_hog_features(image):
    gray = rgb2gray(image)
    return hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True,
    )


def extract_color_histogram(image, bins=32):
    histograms = []

    for channel in range(3):
        hist, _ = np.histogram(image[..., channel], bins=bins, range=(0, 1), density=True)
        histograms.append(hist)

    return np.concatenate(histograms)


def extract_lbp_features(image, points=24, radius=3):
    gray = (rgb2gray(image) * 255).astype(np.uint8)
    lbp = local_binary_pattern(gray, points, radius, method="uniform")
    hist, _ = np.histogram(lbp.ravel(), bins=points + 2, range=(0, points + 2), density=True)
    return hist


def extract_features(image):
    return np.concatenate([
        extract_hog_features(image),
        extract_color_histogram(image),
        extract_lbp_features(image),
    ])


def build_feature_matrix(image_files):
    features = []
    labels = []

    for index, (image_path, label) in enumerate(image_files, start=1):
        image = read_rgb_image(image_path)
        features.append(extract_features(image))
        labels.append(label)

        if index % 25 == 0 or index == len(image_files):
            print(f"Processed {index}/{len(image_files)} images")

    return np.vstack(features), np.array(labels)


X, y_text = build_feature_matrix(image_files)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)

print("Feature matrix:", X.shape)
print("Encoded classes:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

## 5. Train/Test Split

In [ ]:
indices = np.arange(len(y))

X_train, X_test, y_train, y_test, train_indices, test_indices = train_test_split(
    X,
    y,
    indices,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train:", X_train.shape, "Test:", X_test.shape)

## 6. Train SVM

The pipeline standardizes features before fitting the SVM. Grid search chooses useful `C` and `gamma` values for an RBF kernel.

In [ ]:
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", class_weight="balanced")),
])

param_grid = {
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": ["scale", 0.01, 0.001],
}

min_train_class_count = np.bincount(y_train).min()
n_splits = min(5, min_train_class_count)

if n_splits < 2:
    raise ValueError("Each class needs at least 2 training images for cross-validation.")

cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    svm_pipeline,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print(f"Best CV accuracy: {grid_search.best_score_:.3f}")

## 7. Evaluate

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print(f"Test accuracy: {accuracy_score(y_test, y_pred):.3f}")
print()
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=label_encoder.classes_,
    xticks_rotation=45,
    cmap="Blues",
    ax=ax,
)
plt.title("SVM Confusion Matrix")
plt.tight_layout()

## 8. Inspect Mistakes

In [ ]:
mistakes = [
    (idx, true, pred)
    for idx, true, pred in zip(test_indices, y_test, y_pred)
    if true != pred
]

print(f"Mistakes: {len(mistakes)}")

if mistakes:
    sample_mistakes = mistakes[: min(8, len(mistakes))]
    cols = min(4, len(sample_mistakes))
    rows = int(np.ceil(len(sample_mistakes) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3.5 * cols, 3.5 * rows))
    axes = np.atleast_1d(axes).ravel()

    for ax, (idx, true, pred) in zip(axes, sample_mistakes):
        image_path, _ = image_files[idx]
        ax.imshow(read_rgb_image(image_path))
        ax.set_title(
            f"true: {label_encoder.inverse_transform([true])[0]}\n"
            f"pred: {label_encoder.inverse_transform([pred])[0]}"
        )
        ax.axis("off")

    for ax in axes[len(sample_mistakes):]:
        ax.axis("off")

    plt.tight_layout()

## 9. Save and Reuse the Model

In [ ]:
model_bundle = {
    "model": best_model,
    "label_encoder": label_encoder,
    "image_size": IMAGE_SIZE,
}

joblib.dump(model_bundle, MODEL_PATH)
print(f"Saved model to {MODEL_PATH.resolve()}")

In [ ]:
def predict_image(image_path, model_bundle=model_bundle):
    image = read_rgb_image(Path(image_path), image_size=model_bundle["image_size"])
    features = extract_features(image).reshape(1, -1)
    encoded_prediction = model_bundle["model"].predict(features)
    label = model_bundle["label_encoder"].inverse_transform(encoded_prediction)[0]
    return label


# Example:
# predict_image("data/train/class_a/image_001.jpg")

## Notes for Improvement

- Increase `IMAGE_SIZE` if small details matter.
- Use more values in `param_grid` for stronger tuning.
- Try `kernel="linear"` if the dataset is large or feature extraction is already highly discriminative.
- If class folders are imbalanced, keep `class_weight="balanced"` enabled.
- For small datasets, use cross-validation scores as the main signal and treat test accuracy carefully.